# Lab 9: Real CDC with Debezium - Solution

This notebook provides the complete solution for Lab 9, demonstrating how to implement real Change Data Capture (CDC) using Debezium with Iceberg.

## Setup

Configure Spark with Iceberg and CDC support.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Create Spark session with Iceberg support
spark = SparkSession.builder \
    .appName("CDC-Iceberg-Solution") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://polaris:8181/api/catalog") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.streaming.checkpointLocation", "s3a://spark-checkpoints/cdc") \
    .getOrCreate()

## Part 1: Create CDC Tables in Iceberg

In [ ]:
# Create customers CDC table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.cdc.customers (
    id INT,
    name STRING,
    email STRING,
    phone STRING,
    address STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    __op STRING,
    __ts_ms TIMESTAMP,
    __source STRING,
    __table STRING
)
USING iceberg
PARTITIONED BY (days(__ts_ms))
""")

# Create products CDC table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.cdc.products (
    id INT,
    name STRING,
    description STRING,
    price DECIMAL(10, 2),
    category STRING,
    stock_quantity INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    __op STRING,
    __ts_ms TIMESTAMP,
    __source STRING,
    __table STRING
)
USING iceberg
PARTITIONED BY (category)
""")

# Create orders CDC table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.cdc.orders (
    id INT,
    customer_id INT,
    order_date TIMESTAMP,
    status STRING,
    total_amount DECIMAL(10, 2),
    shipping_address STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    __op STRING,
    __ts_ms TIMESTAMP,
    __source STRING,
    __table STRING
)
USING iceberg
PARTITIONED BY (days(order_date))
""")

print("CDC tables created successfully")

## Part 2: Simulate CDC Events

Since we can't directly connect to Debezium from this notebook, we'll simulate CDC events that would come from Debezium.

In [ ]:
# Simulate CDC event structure (Debezium format)
def generate_cdc_event(operation, data, table):
    """Generate a CDC event in Debezium format"""
    return {
        'before': data if operation in ['u', 'd'] else None,
        'after': data if operation in ['c', 'r', 'u'] else None,
        'op': operation,  # c=create, r=read/snapshot, u=update, d=delete
        'ts_ms': int(datetime.utcnow().timestamp() * 1000),
        'source': {
            'version': '2.5.0.Final',
            'connector': 'mysql',
            'name': 'cdc-mysql',
            'ts_ms': int(datetime.utcnow().timestamp() * 1000),
            'snapshot': 'false',
            'db': 'cdc_source',
            'table': table,
            'server_id': 184054
        }
    }

# Sample customer data
customer_data = {
    'id': 1,
    'name': 'John Doe',
    'email': 'john@example.com',
    'phone': '+1-555-0101',
    'address': '123 Main St, NYC, NY 10001',
    'created_at': '2024-01-15T10:30:00',
    'updated_at': '2024-01-15T10:30:00'
}

# Generate CDC events
cdc_events = []

# Create event
cdc_events.append(generate_cdc_event('c', customer_data, 'customers'))

# Update event
updated_customer = customer_data.copy()
updated_customer['phone'] = '+1-555-0102'
updated_customer['updated_at'] = '2024-01-16T14:45:00'
cdc_events.append(generate_cdc_event('u', updated_customer, 'customers'))

# Delete event
cdc_events.append(generate_cdc_event('d', customer_data, 'customers'))

print(f"Generated {len(cdc_events)} CDC events")

## Part 3: Process CDC Events

In [ ]:
# Convert CDC events to DataFrame
cdc_df = spark.createDataFrame(cdc_events)

# Parse CDC events
def parse_cdc_event(event):
    """Parse a CDC event and extract relevant fields"""
    operation = event['op']
    source = event['source']
    ts_ms = event['ts_ms']
    
    if operation in ['c', 'r', 'u']:
        data = event['after']
    else:
        data = event['before']
    
    return {
        'id': data['id'],
        'name': data['name'],
        'email': data['email'],
        'phone': data['phone'],
        'address': data['address'],
        'created_at': data['created_at'],
        'updated_at': data['updated_at'],
        '__op': operation,
        '__ts_ms': datetime.fromtimestamp(ts_ms / 1000),
        '__source': source['name'],
        '__table': source['table']
    }

# Parse all events
parsed_events = [parse_cdc_event(event) for event in cdc_events]
parsed_df = spark.createDataFrame(parsed_events)

print("Parsed CDC events:")
parsed_df.show(truncate=False)

## Part 4: Apply CDC Changes to Iceberg

In [ ]:
# Write all CDC events to Iceberg
parsed_df.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("iceberg.cdc.customers")

print("CDC events written to Iceberg")

## Part 5: Apply CDC Changes (MERGE Pattern)

In [ ]:
# Create a target table for applying changes
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.cdc.customers_target (
    id INT,
    name STRING,
    email STRING,
    phone STRING,
    address STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
""")

# Apply CDC changes using MERGE
# First, handle inserts
inserts = parsed_df.filter(col("__op") == "c")

if inserts.count() > 0:
    # Create staging view for inserts
    inserts.createOrReplaceTempView("cdc_inserts")
    
    spark.sql("""
    MERGE INTO iceberg.cdc.customers_target AS target
    USING cdc_inserts AS source
    ON target.id = source.id
    WHEN NOT MATCHED THEN INSERT *
    """)
    print("Applied inserts")

# Handle updates
updates = parsed_df.filter(col("__op") == "u")

if updates.count() > 0:
    updates.createOrReplaceTempView("cdc_updates")
    
    spark.sql("""
    MERGE INTO iceberg.cdc.customers_target AS target
    USING cdc_updates AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET
        name = source.name,
        email = source.email,
        phone = source.phone,
        address = source.address,
        updated_at = source.updated_at
    """)
    print("Applied updates")

# Handle deletes
deletes = parsed_df.filter(col("__op") == "d")

if deletes.count() > 0:
    deletes.createOrReplaceTempView("cdc_deletes")
    
    spark.sql("""
    DELETE FROM iceberg.cdc.customers_target
    WHERE id IN (SELECT id FROM cdc_deletes)
    """)
    print("Applied deletes")

## Part 6: Verify Applied Changes

In [ ]:
# Check target table
print("Target table state:")
spark.sql("SELECT * FROM iceberg.cdc.customers_target").show()

## Part 7: CDC Data Quality Checks

In [ ]:
# Data quality metrics
quality_metrics = spark.sql("""
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT id) as unique_records,
    COUNT(DISTINCT __op) as operation_types,
    SUM(CASE WHEN __op = 'c' THEN 1 ELSE 0 END) as creates,
    SUM(CASE WHEN __op = 'u' THEN 1 ELSE 0 END) as updates,
    SUM(CASE WHEN __op = 'd' THEN 1 ELSE 0 END) as deletes,
    MIN(__ts_ms) as first_event,
    MAX(__ts_ms) as last_event
FROM iceberg.cdc.customers
""")

print("CDC Data Quality Metrics:")
quality_metrics.show()

## Part 8: CDC Operation Distribution

In [ ]:
# Operation distribution
op_distribution = spark.sql("""
SELECT 
    __op as operation,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM iceberg.cdc.customers
GROUP BY __op
ORDER BY count DESC
""")

print("CDC Operation Distribution:")
op_distribution.show()

## Part 9: CDC Timeline Analysis

In [ ]:
# Timeline analysis
timeline = spark.sql("""
SELECT 
    DATE(__ts_ms) as event_date,
    __op as operation,
    COUNT(*) as event_count
FROM iceberg.cdc.customers
GROUP BY DATE(__ts_ms), __op
ORDER BY event_date, operation
""")

print("CDC Timeline:")
timeline.show()

## Part 10: Schema Evolution with CDC

In [ ]:
# Add new column to handle schema evolution
spark.sql("""
ALTER TABLE iceberg.cdc.customers 
ADD COLUMN loyalty_points INT
""")

print("Schema evolved successfully")

In [ ]:
# Insert data with new field
customer_with_loyalty = customer_data.copy()
customer_with_loyalty['loyalty_points'] = 100
cdc_events.append(generate_cdc_event('c', customer_with_loyalty, 'customers'))

# Parse and insert
parsed_new = [parse_cdc_event(e) for e in cdc_events[-1:]]
parsed_new_df = spark.createDataFrame(parsed_new)

# Handle missing loyalty_points for old records
parsed_new_df = parsed_new_df.withColumn(
    "loyalty_points", coalesce(col("loyalty_points"), lit(0))
)

parsed_new_df.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("iceberg.cdc.customers")

print("Schema evolution data inserted successfully")

## Part 11: Monitor CDC Snapshots

In [ ]:
# Check table snapshots
snapshots = spark.sql("CALL iceberg.system.history('iceberg.cdc.customers')")
print("CDC Table Snapshots:")
snapshots.show()

## Cleanup

In [ ]:
# Drop CDC tables
spark.sql("DROP TABLE IF EXISTS iceberg.cdc.customers")
spark.sql("DROP TABLE IF EXISTS iceberg.cdc.customers_target")
spark.sql("DROP TABLE IF EXISTS iceberg.cdc.products")
spark.sql("DROP TABLE IF EXISTS iceberg.cdc.orders")

print("CDC tables dropped successfully")

## Summary

This solution demonstrated:

1. **CDC Table Creation**: Created Iceberg tables with CDC metadata fields
2. **CDC Event Simulation**: Simulated Debezium CDC events with proper format
3. **Event Processing**: Parsed and processed CDC events with operation types
4. **Change Application**: Applied CDC changes using MERGE operations
5. **Data Quality**: Implemented CDC-specific data quality checks
6. **Operation Tracking**: Tracked create, update, and delete operations
7. **Timeline Analysis**: Analyzed CDC events over time
8. **Schema Evolution**: Demonstrated handling schema changes in CDC
9. **Monitoring**: Added snapshot monitoring for CDC tables

In a production environment, you would:
- Connect to actual Debezium connectors via Kafka
- Implement continuous streaming processing
- Add proper error handling and retry logic
- Implement exactly-once processing semantics
- Add monitoring and alerting for CDC pipeline health
- Handle schema evolution automatically
- Implement SCD (Slowly Changing Dimensions) patterns